# Importing libraries

In [1]:
import os
import requests
from pathlib import Path
import pandas as pd

# Basic wrangling and external features
import joblib
import json
import pickle
import numpy as np
import holidays
import osmnx as ox
import scipy
from scipy import stats
from datetime import datetime, timedelta

# For visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import folium

# For maps
import geopandas as gpd
from shapely.geometry import Point
import contextily as cx

# For linear models
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import PoissonRegressor

#For tree-based models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
 
#For time series
#from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
#from statsmodels.tsa.stattools import adfuller

/Users/francobastida/miniconda3/envs/ecobici/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


# Prediction model panel build up

In [2]:
# Station-hour panel parquet 
DATA_PATH = Path("../outputs/ecobici_station_hour.parquet")

df = pd.read_parquet(DATA_PATH)
print(df.shape)
df.head()

(22463807, 14)


,station_id,datetime_hour,departures,arrivals,net_flow,abs_net_flow,year,month,day,hour,weekday,is_weekend,is_morning_peak,is_evening_peak
0,1,2018-01-01,0,2,-2,2,2018,1,1,0,1,0,0,0
1,6,2018-01-01,3,1,2,2,2018,1,1,0,1,0,0,0
2,9,2018-01-01,1,0,1,1,2018,1,1,0,1,0,0,0
3,16,2018-01-01,7,0,7,7,2018,1,1,0,1,0,0,0
4,18,2018-01-01,14,0,14,14,2018,1,1,0,1,0,0,0


In [ ]:
# Loading
weather_df = pd.read_parquet("../data/reference/weather_hourly_cdmx_2018_2025.parquet")
holiday_df = pd.read_parquet("../data/reference/mx_holidays.parquet")
multimodal_df = pd.read_parquet("../data/reference/stations_gbfs_with_metro_distance.parquet")

In [4]:
weather_df

,datetime_hour,temperature,rain,is_raining
0,2018-01-01 00:00:00,13.2,0.0,0
1,2018-01-01 01:00:00,12.1,0.0,0
2,2018-01-01 02:00:00,11.0,0.0,0
3,2018-01-01 03:00:00,10.1,0.0,0
4,2018-01-01 04:00:00,9.2,0.0,0
...,...,...,...,...
70123,2025-12-31 19:00:00,18.1,0.0,0
70124,2025-12-31 20:00:00,17.2,0.0,0
70125,2025-12-31 21:00:00,16.5,0.0,0
70126,2025-12-31 22:00:00,16.2,0.0,0


In [5]:
# Pre merge standardization
df["datetime_hour"] = pd.to_datetime(df["datetime_hour"])
weather_df["datetime_hour"] = pd.to_datetime(weather_df["datetime_hour"])

df["station_id"] = df["station_id"].astype(str).str.strip()
stations_df["station_id"] = stations_df["station_id"].astype(str).str.strip()
multimodal_df["station_id"] = multimodal_df["station_id"].astype(str).str.strip()

df["date"] = df["datetime_hour"].dt.date
holiday_df["date"] = pd.to_datetime(holiday_df["date"]).dt.date

In [ ]:
# Weather features
df = df.merge(
    weather_df[["datetime_hour", "temperature", "rain", "is_raining"]],
    on="datetime_hour",
    how="left",
    validate="many_to_one"
)

print(df.columns[df.columns.str.endswith(("_x", "_y"))].tolist())

[]


In [12]:
holiday_df

,date,is_holiday,holiday_name
0,2018-01-01,1,New Year's Day
1,2018-01-02,0,None
2,2018-01-03,0,None
3,2018-01-04,0,None
4,2018-01-05,0,None
...,...,...,...
2917,2025-12-27,0,None
2918,2025-12-28,0,None
2919,2025-12-29,0,None
2920,2025-12-30,0,None


In [13]:
# Holiday features
df = df.merge(
    holiday_df[["date", "is_holiday", "holiday_name"]],
    on="date",
    how="left",
    validate="many_to_one"
)


In [18]:
# Multimodal features
df = df.merge(
    multimodal_df[["station_id", "lat", "lon", "capacity", "nearest_metro_name", "distance_to_metro_m"]],
    on="station_id",
    how="left",
    validate="many_to_one"
)


In [22]:
df

,station_id,datetime_hour,departures,arrivals,net_flow,abs_net_flow,year,month,day,hour,...,temperature,rain,is_raining,is_holiday,holiday_name,lat,lon,capacity,nearest_metro_name,distance_to_metro_m
0,1,2018-01-01 00:00:00,0,2,-2,2,2018,1,1,0,...,13.2,0.0,0,1,New Year's Day,19.416795,-99.192508,39.0,Constituyentes,573.702169
1,6,2018-01-01 00:00:00,3,1,2,2,2018,1,1,0,...,13.2,0.0,0,1,New Year's Day,19.363404,-99.160395,27.0,Parque de los Venados,822.140398
2,9,2018-01-01 00:00:00,1,0,1,1,2018,1,1,0,...,13.2,0.0,0,1,New Year's Day,19.367816,-99.175269,23.0,Insurgentes Sur,636.962261
3,16,2018-01-01 00:00:00,7,0,7,7,2018,1,1,0,...,13.2,0.0,0,1,New Year's Day,19.370566,-99.156622,27.0,Parque de los Venados,239.823584
4,18,2018-01-01 00:00:00,14,0,14,14,2018,1,1,0,...,13.2,0.0,0,1,New Year's Day,19.372474,-99.171639,35.0,Hospital 20 de Noviembre,113.424432
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22463802,677,2025-12-31 23:00:00,0,1,-1,1,2025,12,31,23,...,15.5,0.0,0,0,None,NaN,NaN,NaN,NaN,NaN
22463803,688,2025-12-31 23:00:00,2,0,2,2,2025,12,31,23,...,15.5,0.0,0,0,None,NaN,NaN,NaN,NaN,NaN
22463804,696,2025-12-31 23:00:00,0,1,-1,1,2025,12,31,23,...,15.5,0.0,0,0,None,NaN,NaN,NaN,NaN,NaN
22463805,697,2025-12-31 23:00:00,0,2,-2,2,2025,12,31,23,...,15.5,0.0,0,0,None,19.463941,-99.182280,27.0,Cuitláhuac,750.536947


In [25]:
# Merge check based on full parquet
print(df["lat"].isna().mean())
print(df["lon"].isna().mean())
print(df["capacity"].isna().mean())
print(df["nearest_metro_name"].isna().mean())
print(df["distance_to_metro_m"].isna().mean())

0.031712033494589766
0.031712033494589766
0.031712033494589766
0.031712033494589766
0.031712033494589766


- After the last merge, about 3% of rows are negligible based on the station_id data
- These are mostly likely either discontinued or missing from the GBFS subset
- For modeling purposes, these stations have been dropped to prevent biases or leakage

In [26]:
df = df.dropna(subset=["lat", "lon", "capacity", "nearest_metro_name", "distance_to_metro_m"])

In [33]:
print(df.shape)
df.head()

(21751434, 25)


,station_id,datetime_hour,departures,arrivals,net_flow,abs_net_flow,year,month,day,hour,...,temperature,rain,is_raining,is_holiday,holiday_name,lat,lon,capacity,nearest_metro_name,distance_to_metro_m
0,1,2018-01-01,0,2,-2,2,2018,1,1,0,...,13.2,0.0,0,1,New Year's Day,19.416795,-99.192508,39.0,Constituyentes,573.702169
1,6,2018-01-01,3,1,2,2,2018,1,1,0,...,13.2,0.0,0,1,New Year's Day,19.363404,-99.160395,27.0,Parque de los Venados,822.140398
2,9,2018-01-01,1,0,1,1,2018,1,1,0,...,13.2,0.0,0,1,New Year's Day,19.367816,-99.175269,23.0,Insurgentes Sur,636.962261
3,16,2018-01-01,7,0,7,7,2018,1,1,0,...,13.2,0.0,0,1,New Year's Day,19.370566,-99.156622,27.0,Parque de los Venados,239.823584
4,18,2018-01-01,14,0,14,14,2018,1,1,0,...,13.2,0.0,0,1,New Year's Day,19.372474,-99.171639,35.0,Hospital 20 de Noviembre,113.424432


# Lagging

In [35]:
# Sorting
model_df = df.sort_values(["station_id", "datetime_hour"]).copy()

#Lag creation

model_df["lag_1"] = model_df.groupby("station_id")["departures"].shift(1)
model_df["lag_24"] = model_df.groupby("station_id")["departures"].shift(24)

In [ ]:
# Checking for missingness
model_df.isna().mean().sort_values(ascending=False)

lag_24                 0.000747
lag_1                  0.000031
date                   0.000000
distance_to_metro_m    0.000000
nearest_metro_name     0.000000
capacity               0.000000
lon                    0.000000
lat                    0.000000
holiday_name           0.000000
is_holiday             0.000000
is_raining             0.000000
rain                   0.000000
temperature            0.000000
station_id             0.000000
datetime_hour          0.000000
is_morning_peak        0.000000
is_weekend             0.000000
weekday                0.000000
hour                   0.000000
day                    0.000000
month                  0.000000
year                   0.000000
abs_net_flow           0.000000
net_flow               0.000000
arrivals               0.000000
departures             0.000000
is_evening_peak        0.000000
dtype: float64

- After merging and creating the data set that will be used for modeling, lag_1 has 0.003% NAs and lag_24 0.07%
- These values are potentially crated from the construction of the lagged demand itself between times
- The time alignment is correct and good enough for modeling, so these will be dropped from the dataset


In [39]:
model_df = model_df.dropna()

In [ ]:
# Last check to see that the lags are evenle spaces, exactly 1 hour apart

model_df.groupby("station_id")["datetime_hour"].diff().min()

Timedelta('0 days 01:00:00')

In [41]:
model_df.to_parquet(
    "../outputs/ecobici_prediction_panel.parquet",
    index=False
)